In [ ]:
from pathlib import Path
import subprocess
import sys

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/urdu-question-generator')
    if not ROOT.exists():
        subprocess.run(['git', 'clone', 'https://github.com/Areesha-008/Urdu-Question-Generator-.git', str(ROOT)], check=True)
    BASE = Path('/content/drive/MyDrive/Urdu-QG-v2')
else:
    ROOT = Path.cwd() if (Path.cwd() / 'config.py').exists() else Path.cwd().parent
    BASE = ROOT / 'artifacts'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(ROOT / 'requirements.txt')], check=True)
sys.path.insert(0, str(ROOT))
DATA = BASE / 'data'
RUN = BASE / 'answer_gru_v1'
RUN.mkdir(parents=True, exist_ok=True)
print('Data:', DATA, 'Run:', RUN)


In [ ]:
import json
import numpy as np
from scripts.prepare_data import read_pairs
from scripts.train_tokenizer import train_tokenizer

tokenizer = train_tokenizer(DATA / 'train.tsv', RUN)
statistics = {}
for name in ['train', 'valid', 'wiki_test']:
    pairs = read_pairs(DATA / f'{name}.tsv')
    statistics[name] = {}
    for column, label in enumerate(['source', 'target']):
        lengths = [len(tokenizer.encode(pair[column])) for pair in pairs]
        statistics[name][label] = {
            'percentiles': np.percentile(lengths, [50, 95, 99, 100]).tolist(),
            'fertility': sum(lengths) / sum(len(pair[column].split()) for pair in pairs)}
(RUN / 'tokenizer_statistics.json').write_text(json.dumps(statistics, indent=2))
print(statistics)
for _, target in read_pairs(DATA / 'train.tsv')[:5]:
    print(target, tokenizer.encode(target, pieces=True), sep='\n')
